# Task 1.1 — Core Contribution / Algorithm Architecture (8 marks)

**Paper**: *Robust Point Set Registration Using Gaussian Mixture Models* — Bing Jian and Baba C. Vemuri, IEEE TPAMI, 2011.

---

## Step-by-Step Method Description

### Step 1: Point Set Representation as Gaussian Mixtures
- **Description**: Given two point sets — a *model* set $M = \{m_1, \ldots, m_k\}$ and a *scene* set $S = \{s_1, \ldots, s_n\}$ — each set is represented as a Gaussian mixture model (GMM). Each point becomes the centre of an isotropic Gaussian component with a shared bandwidth parameter $\sigma$. The GMM for the model is: $f(x) = \frac{1}{k} \sum_{i=1}^{k} \phi(x | m_i, \sigma^2 I)$, and similarly for the scene.
- **Reference**: Section 2, Equation 2 of the paper.
- **Purpose**: Converting discrete point sets into continuous density functions allows the registration problem to be formulated as a continuous optimisation problem. This side-steps the hard combinatorial correspondence problem that methods like ICP face.

### Step 2: L2 Distance as the Discrepancy Measure
- **Description**: The discrepancy between the two GMMs is measured using the L2 distance (integrated squared difference): $d_{L2}(f, g) = \int (f(x) - g(x))^2 dx$. This expands into three terms involving integrals of Gaussian products — and crucially, each such integral has a **closed-form** solution: $\int \phi(x|\mu_1, \Sigma_1) \phi(x|\mu_2, \Sigma_2) dx = \phi(0 | \mu_1 - \mu_2, \Sigma_1 + \Sigma_2)$. Since the scene is fixed, the term $\int g^2 dx$ is constant, and only the cross-term and model self-term need to be computed.
- **Reference**: Section 3, Equations 4 and 5.
- **Purpose**: The closed-form expression makes the L2 distance computationally efficient to evaluate and differentiable, which is essential for gradient-based optimisation. The L2 distance is also inherently robust to outliers — its influence function is bounded, unlike the KL divergence used in maximum likelihood estimation (Section 3.1).

### Step 3: Transformation Model (Thin-Plate Splines)
- **Description**: The model point set is transformed using a parameterised transformation $T(\theta)$. For non-rigid registration, the paper uses thin-plate splines (TPS). A set of control points $Q = \{q_1, \ldots, q_c\}$ defines the TPS. The transformation of any point $x$ is: $T(x) = Ax + t + \sum_{j=1}^{c} w_j K(|x - q_j|)$ where $K(r) = r^2 \log r$ in 2D. The full parameter vector $\theta$ consists of the affine parameters $(A, t)$ and the non-linear TPS weights $\{w_j\}$.
- **Reference**: Section 5, Equations 16–17.
- **Purpose**: TPS provides a flexible non-rigid transformation model that can capture smooth deformations. The bending energy of the TPS serves as a natural regulariser to prevent overfitting.

### Step 4: Objective Function with Regularisation
- **Description**: The full objective function combines the L2 distance with a regularisation term: $E(\theta) = d_{L2}(T(M, \theta), S, \sigma) + \lambda \cdot \text{trace}(W^T K W)$, where $W$ are the TPS non-affine weights and $K$ is the TPS kernel matrix. The hyperparameter $\lambda$ controls the trade-off between data fidelity and smoothness. Analytical gradients $\nabla_\theta E$ are derived, enabling the use of quasi-Newton optimisers.
- **Reference**: Section 5, Equation 14 and surrounding discussion.
- **Purpose**: The regularisation term penalises non-smooth deformations, preventing the transformation from overfitting to noise or outliers. The analytical gradient computation is critical for efficiency.

### Step 5: Multi-Scale (Coarse-to-Fine) Optimisation
- **Description**: The algorithm operates in multiple levels. It starts with a large bandwidth $\sigma$ (giving a smooth, broadly peaked objective landscape) and progressively decreases $\sigma$ (revealing finer details). At each level, the L-BFGS-B quasi-Newton optimiser is run with the current $\sigma$ and a corresponding regularisation weight $\lambda_i$, using the solution from the previous level as initialisation.
- **Reference**: Algorithm 1, Section 5.
- **Purpose**: Large $\sigma$ at early stages creates a convex-like landscape that avoids local minima and provides a good coarse alignment. Smaller $\sigma$ at later stages refines the alignment. This is an annealing strategy that is key to achieving global convergence.

### Step 6: Output — Transformed Model
- **Description**: After the multi-level optimisation converges, the optimised parameters $\theta^*$ are used to transform the model points via the TPS. If normalisation was applied at the start, the transformed model is de-normalised to return to the original coordinate frame. The output is the registered model point set aligned to the scene.
- **Reference**: Section 5 (normalisation), `run_config` function in the author's code.
- **Purpose**: Produces the final aligned point set that can be used for downstream tasks like shape analysis, medical image registration, or correspondence estimation.

---

## Final Summary

This paper solves the problem of **rigid and non-rigid point set registration in the presence of noise and outliers** by representing point sets as Gaussian mixtures and minimising the L2 distance between them via multi-scale optimisation; the authors claim their approach is superior to existing methods (especially ICP and EM-based methods) because the L2 distance has an **analytically computable closed-form expression**, provides **inherent statistical robustness** through its bounded influence function, and avoids the need for explicit one-to-one correspondence estimation.